# Week 6 Deliverable - Expanded Feature Engineering

# Revisions before week 6 activities
- Redoing files to include the newest 202606 .csv data set
    - new test month
    - more dynamic data collection
    - rolling collection
- 02 updated to include latitude longitude geographical features earlier on
- new cleaned.csv file generated
- expand dataframe for at least a full year of data (Jun 2025 – Jun 2026)
202506-202605 04_model_comparison (May test month):
- BEST: RandomForest.C R^2 = 0.8136	via GridSearchCV/TimeSeriesSplit(4), LOG target, ... 
- Introduced the newest data of 06 2026
- fixed training window discrepancy between 03_baseline and 04_model_comparison
    - both now cap at 11 train months (Jul 2025 – May 2026) to match
202506-202606 04_model_comparison (June test month):
    - BEST: RandomForest.A R^2 0.775 via unconstrained defaults, log target
- switching to a different test month has a slight variation between resulting values
- updated markdowns in all files to reflect edits
- added residual plots (RF.A) and a train-vs-test gap visualization

# Week 6 Feature Engineering

Task:
- Example of sample features you can engineer: bed/bath ratio, age of property in years
- Adding more detailed geographic layer using school districts: build a more detailed regional feature by spatially joining each property’s coordinates against the CA School District Areas 2025-26 boundaries
    - Source: https://data.ca.gov/dataset/california-school-district-areas-2025-26 
- Re-train models with the updated feature set.
- Deliverable: Updated notebook + table comparing old vs new feature sets, including the school district layer.

New features were implemented in `02_preprocessing.ipynb` and tested in `04_model_comparison` for overall impact in models.

## School District Mapping
- installed geopandas to local machine
- downloaded 2025-26 GeoJSON, stored in /data (.gitignored) not uploaded to github
    - https://data.ca.gov/dataset/california-school-district-areas-2025-26
    - .../data/DistrictAreas2526_-284845464123469011.geojson

- After filtering:
No_Unified_District     34513
Los Angeles Unified     14193
San Diego Unified        3864
Capistrano Unified       2637
Desert Sands Unified     2606
Palm Springs Unified     2322
Oakland Unified          1859
Corona-Norco Unified     1774
Long Beach Unified       1767
Hemet Unified            1697
- high amount of non unified districts means those areas will need a different geographical lead: CountyOrParish
- Cross-checked the spatial join against the raw HighSchoolDistrict on the 6.5% of rows where it's present
    - they agree on genuinely-unified areas

## Bed/Bath Ratio
BedperBath: min 0.14 | median 1.50 | max 7.50

## Property Age
PropertyAge: min 0 | median 49 | max 172
negatives clipped to 0: 14
- 14 listed properties made in less than a year, set to 0

## LotSizeAcres to LivingArea ratio
- investigating the land to actual livable area of the property

LotToLiving: min 0.10 | median 4.22 | max 20808.98
  p95: 22.5 | p99: 134.8
LotToLiving_log: min -2.30 | median 1.44 | max 9.94
  p95: 3.11 | p99: 4.90

# Testing against the models

- Efficient route: Comparing performance of the original Linear Regression model vs the winning Random Forest model

 model          feature_set  n_features  train_r2  test_r2     gap
Linear             Original          73    0.5097   0.5623 -0.0525
Linear          +engineered          76    0.0929   0.5612 -0.4683
Linear            +district         436    0.6984   0.7120 -0.0136
Linear +engineered+district         439    0.6529   0.7383 -0.0855
          model          feature_set  n_features  train_r2  test_r2    gap  vs_baseline
Random Forest A             Original          73    0.9484   0.7750 0.1734       0.2127
Random Forest A          +engineered          76    0.9458   0.7558 0.1900       0.1935
Random Forest A            +district         436    0.9494   0.7806 0.1688       0.2183
Random Forest A +engineered+district         439    0.9473   0.7673 0.1800       0.2050

- Linear improved immensely with the addition of all the new feature engineering including the school district.
- Random Forest model A still wins over the Linear regression value with slight improvement due to the addition of the School District layer.

# Week 6 result

- Running each addition separately (engineered on/off, district on/off) on both the random forest and the linear baseline shows the two models respond in opposite directions to the same features:

Random Forest:
- School-district layer helps improve a little raising R^2 by +0.0056
    - Result: best model overall (0.7806), while tightening the train/test gap (0.173 → 0.169) 
    - Extra geographical layer

- Engineered ratios added individually on top of district, every one lowers test R^2 (BedperBath −0.016, PropertyAge −0.005, LotToLiving_log −0.002)
    - R^2: 0.7806 to 0.7673 
    - They are lossier recombinations of features the forest already has, and a tree discovers these interactions on its own.

*Linear baseline:*
- **The same features that hurt RF help Linear.** District alone lifts it +0.150 (0.5623 → 0.7120), and adding the engineered ratios lifts it further to 0.7383 — its best configuration. Linear cannot represent geography or interactions internally, so explicit district categories and engineered ratios give it signal it otherwise lacks.

**Why the split:** the random forest already captures location through lat/long and its ability to partition the coordinate plane, so district and engineered features are largely redundant for it. Linear can do neither, so the same features are transformative. Feature engineering pays off for weaker model classes and diminishes as the model becomes more expressive — which also explains the teammate gap, since their geographic-feature gains came from improving simpler models from a lower base.

**Decision:** adopt **Original + district (RF, 0.7806)** as the Week 6 production model; drop the engineered ratios (they only help the model we aren't shipping). Verified that regularization doesn't change this — RF.B + district (0.7567, gap 0.090) gains almost nothing from district and stays the tighter-gap alternative, so the RF.A-vs-RF.B tradeoff is unchanged.